# Does $\pi^{\mathrm{hyb}} = \pi^{\mathrm{GPI}}_{\mathcal B}$ in general? (Conjecture 1 from `notes/adaptive_basis.tex`)

`notes/adaptive_basis.tex` recorded a Conjecture, based on exhaustive search over Office
(k=1..6, 150 weights) and a walled maze (k=1,2, 150 weights): that the hybrid bootstrap
policy $\pi^{\mathrm{hyb}}$ (one full-action-set greedy step off the metapolicy's own
converged value $V_{\mathrm{meta}}$) always realizes exactly the same value as ordinary
GPI over the base $\mathcal B$, $\pi^{\mathrm{GPI}}_{\mathcal B}$.

Both prior searches only covered **deterministic, terminate-on-pickup navigation
domains** -- Office, DST, a walled maze. This notebook tests the conjecture on
**generic random MDPs** (both deterministic and stochastic, both dense and sparse
one-hot rewards, no special navigation structure) via exact linear algebra throughout
(no learning, no approximation -- any discrepancy found is a real counterexample, not
numerical noise, once floating-point tie artifacts are ruled out below).

**Summary of what this notebook finds:**
1. At $k=1$, the tie is an exact algebraic identity, not a coincidence (Lemma + Theorem
   below) -- confirmed with 0/600 violations across every tested configuration.
2. At $k\ge2$, the conjecture is **false**: concrete counterexamples exist in MDPs as
   small as 3 states, 3-4 actions, 2 objectives -- in *both* directions ($\pi^{\mathrm{hyb}}$
   strictly beats $\pi^{\mathrm{GPI}}_{\mathcal B}$ in one example, loses in another).
3. A "terminate on pickup" absorbing structure (mirroring Office/DST/the maze) reduces
   the violation rate roughly $10\times$ but does not eliminate it -- so the earlier
   zero-counterexample findings look like a small-sample coincidence, not a structural
   guarantee.

## 1. Exact solvers for general finite MDPs (deterministic or stochastic transitions)

In [1]:
import numpy as np

np.set_printoptions(suppress=True, precision=4)

GAMMA = 0.9


def make_mdp(n, m, d, rng, deterministic, sparse_reward, jitter=1e-3):
    """A random finite MDP: n states, m actions everywhere, d objectives.
    deterministic=False draws a genuinely stochastic Dirichlet transition kernel.
    sparse_reward=True mimics a "pickup" domain (each (s,a) triggers at most one
    objective's reward, matching phi's structure in DST/Office/the maze); jitter
    breaks the many exact Q-value ties that a clean sparse/one-hot reward otherwise
    creates, so that any argmax discrepancy found reflects a real inequality rather
    than floating-point noise resolving a genuine tie in opposite directions."""
    if deterministic:
        nxt = rng.integers(0, n, size=(n, m))
        P = None
    else:
        nxt = None
        alpha = 0.3
        P = np.stack([rng.dirichlet(alpha * np.ones(n), size=m) for _ in range(n)], axis=0)

    if sparse_reward:
        phi = np.zeros((n, m, d))
        mask = rng.random((n, m)) < 0.2
        idx = rng.integers(0, d, size=(n, m))
        for s in range(n):
            for a in range(m):
                if mask[s, a]:
                    phi[s, a, idx[s, a]] = 1.0
    else:
        phi = rng.random((n, m, d))
    phi = phi + jitter * rng.random((n, m, d))
    return nxt, P, phi


def trans_row(s, a, nxt, P, n):
    if nxt is not None:
        row = np.zeros(n)
        row[nxt[s, a]] = 1.0
        return row
    return P[s, a, :]


def build_Ppi(pi, nxt, P, n):
    Ppi = np.zeros((n, n))
    for s in range(n):
        Ppi[s, :] = trans_row(s, pi[s], nxt, P, n)
    return Ppi


def evaluate_scalar(pi, nxt, P, phi_w, gamma, n):
    Ppi = build_Ppi(pi, nxt, P, n)
    r_pi = np.array([phi_w[s, pi[s]] for s in range(n)])
    return np.linalg.solve(np.eye(n) - gamma * Ppi, r_pi)


def evaluate_vector(pi, nxt, P, phi, gamma, n, d):
    Ppi = build_Ppi(pi, nxt, P, n)
    r_pi = np.stack([phi[s, pi[s], :] for s in range(n)], axis=0)
    return np.linalg.solve(np.eye(n) - gamma * Ppi, r_pi)


def full_sf_table(pi, nxt, P, phi, gamma, n, m, d):
    """Psi(s,a) for the FIXED policy pi, evaluated at every (s,a): take a once, then
    follow pi's own greedy action from the landing state onward."""
    V_vec = evaluate_vector(pi, nxt, P, phi, gamma, n, d)
    Psi = np.zeros((n, m, d))
    for s in range(n):
        for a in range(m):
            row = trans_row(s, a, nxt, P, n)
            Psi[s, a, :] = phi[s, a, :] + gamma * (row @ V_vec)
    return Psi, V_vec


def policy_iteration(phi_w, nxt, P, gamma, n, m, allowed=None, max_iter=1000):
    """Exact policy iteration, optionally restricted to `allowed[s]` actions per
    state -- this is Definition 2's T_B when `allowed` is the k base policies' own
    actions, and ordinary unrestricted PI when `allowed=None`."""
    if allowed is None:
        allowed = [np.arange(m) for _ in range(n)]
    pi = np.array([allowed[s][0] for s in range(n)])
    for _ in range(max_iter):
        V = evaluate_scalar(pi, nxt, P, phi_w, gamma, n)
        new_pi = pi.copy()
        for s in range(n):
            best_a, best_q = None, -np.inf
            for a in allowed[s]:
                row = trans_row(s, a, nxt, P, n)
                q = phi_w[s, a] + gamma * (row @ V)
                if q > best_q + 1e-12:
                    best_q, best_a = q, a
            new_pi[s] = best_a
        if np.array_equal(new_pi, pi):
            break
        pi = new_pi
    return pi, evaluate_scalar(pi, nxt, P, phi_w, gamma, n)


def full_greedy(Qsa):
    return np.argmax(Qsa, axis=1)


def solve_basis_and_compare(nxt, P, phi, base_pis, w, gamma, n, m, d):
    """Given an already-solved basis (list of policies), compute, for query weight w:
    metapolicy (restricted PI), pi_hyb (full-action greedy off V_meta), and
    pi_gpi (full-action greedy off the pool estimate Q-hat). Returns realized scalar
    values V_meta, V_hyb, V_gpi (each length n) plus the three policies."""
    k = len(base_pis)
    base_psis = [full_sf_table(pi_j, nxt, P, phi, gamma, n, m, d)[0] for pi_j in base_pis]
    phi_w = phi @ w

    allowed = [np.unique([base_pis[j][s] for j in range(k)]) for s in range(n)]
    pi_meta, V_meta = policy_iteration(phi_w, nxt, P, gamma, n, m, allowed=allowed)

    Qhat = np.zeros((n, m))
    for s in range(n):
        for a in range(m):
            Qhat[s, a] = max(base_psis[j][s, a, :] @ w for j in range(k))
    pi_gpi = full_greedy(Qhat)
    V_gpi = evaluate_scalar(pi_gpi, nxt, P, phi_w, gamma, n)

    Q_hybsrc = np.zeros((n, m))
    for s in range(n):
        for a in range(m):
            row = trans_row(s, a, nxt, P, n)
            Q_hybsrc[s, a] = phi_w[s, a] + gamma * (row @ V_meta)
    pi_hyb = full_greedy(Q_hybsrc)
    V_hyb = evaluate_scalar(pi_hyb, nxt, P, phi_w, gamma, n)

    return dict(pi_meta=pi_meta, pi_gpi=pi_gpi, pi_hyb=pi_hyb,
                V_meta=V_meta, V_gpi=V_gpi, V_hyb=V_hyb, Qhat=Qhat, Q_hybsrc=Q_hybsrc)


## 2. The $k=1$ identity, exactly

$\Psi^{\pi_{\mathrm{meta}}}(s,a)^\top w = \phi(s,a)^\top w + \gamma\,\mathbb E_{s'}[V_{\mathrm{meta}}(s')]$
holds for **every** $a$ (not just $a=\pi_{\mathrm{meta}}(s)$), by the successor-feature
Bellman recursion applied to the *fixed* stationary policy $\pi_{\mathrm{meta}}$. At $k=1$,
$\widehat Q_{\mathcal B}$ collapses to exactly this same function (the pool has one member),
so $\pi^{\mathrm{hyb}}$ and $\pi^{\mathrm{GPI}}_{\mathcal B}$ are argmaxes of the *identical*
real-valued function -- forced to agree.

In [2]:
rng = np.random.default_rng(0)

n, m, d, gamma = 6, 4, 3, GAMMA
nxt, P, phi = make_mdp(n, m, d, rng, deterministic=True, sparse_reward=True)

w_train = rng.dirichlet(np.ones(d))
pi1, _ = policy_iteration(phi @ w_train, nxt, P, gamma, n, m)
Psi1, V_vec = full_sf_table(pi1, nxt, P, phi, gamma, n, m, d)

w = rng.dirichlet(np.ones(d))
phi_w = phi @ w
allowed = [np.array([pi1[s]]) for s in range(n)]
pi_meta, V_meta = policy_iteration(phi_w, nxt, P, gamma, n, m, allowed=allowed)

Qhat = Psi1 @ w
Q_hybsrc = np.zeros((n, m))
for s in range(n):
    for a in range(m):
        row = trans_row(s, a, nxt, P, n)
        Q_hybsrc[s, a] = phi_w[s, a] + gamma * (row @ V_meta)

print("Lemma check: Psi^pi_meta(s,a).w == phi(s,a).w + gamma*E[V_meta(s')] for ALL a, k=1")
print("max |Qhat - Q_hybsrc| =", np.max(np.abs(Qhat - Q_hybsrc)), "(machine precision)")
print()
print("pi_hyb  =", full_greedy(Q_hybsrc))
print("pi_gpi  =", full_greedy(Qhat))
print("identical policies -> identical realized value, exactly, at k=1.")


Lemma check: Psi^pi_meta(s,a).w == phi(s,a).w + gamma*E[V_meta(s')] for ALL a, k=1
max |Qhat - Q_hybsrc| = 6.661338147750939e-16 (machine precision)

pi_hyb  = [1 2 1 2 3 3]
pi_gpi  = [1 2 1 2 3 3]
identical policies -> identical realized value, exactly, at k=1.


## 3. A methodological trap: floating-point ties can fake a $k=1$ "violation"

Since the identity above is exact only in real arithmetic, computing its two sides via
different linear solves can round oppositely at a genuine value-tie, flipping the argmax
and producing a large *apparent* discrepancy that has nothing to do with the underlying
mathematics. This bit us on the first pass: an early sweep with clean (non-jittered)
sparse rewards reported $k=1$ "violations" up to $2.78$. Reproduced and explained below;
this is why every sweep in this notebook adds a small `1e-3` reward jitter.

In [3]:
print("Demonstrating the floating-point tie artifact at k=1 (a fixed, clean, non-jittered example)")
print()

nxt = np.array([[1, 0, 2, 1], [0, 0, 3, 5], [4, 1, 3, 2], [3, 4, 5, 3], [3, 0, 4, 4], [5, 0, 1, 0]])
phi = np.array([
    [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.0, 0.0]],
    [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
    [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
])
w = np.array([0.07726823409979573, 0.7834612875183348, 0.13927047838186965])
pi1 = np.array([0, 2, 0, 1, 3, 2])
n, m, d, gamma = 6, 4, 3, GAMMA
P = None

Psi1, _ = full_sf_table(pi1, nxt, P, phi, gamma, n, m, d)
phi_w = phi @ w
allowed = [np.array([pi1[s]]) for s in range(n)]
pi_meta, V_meta = policy_iteration(phi_w, nxt, P, gamma, n, m, allowed=allowed)
Qhat = Psi1 @ w
Q_hybsrc = np.zeros((n, m))
for s in range(n):
    for a in range(m):
        row = trans_row(s, a, nxt, P, n)
        Q_hybsrc[s, a] = phi_w[s, a] + gamma * (row @ V_meta)

with np.printoptions(precision=17, suppress=False):
    print("row 0 of Qhat,     full precision:", Qhat[0])
    print("row 0 of Q_hybsrc, full precision:", Q_hybsrc[0])
print("Same real-valued function (Lemma 1) -- a genuine 3-way tie among actions {0,2,3}")
print("at value 0.62587... -- but rounded via two different linear solves, so argmax flips:")
print("argmax(Qhat)     =", full_greedy(Qhat), " (picks action 0 at s=0)")
print("argmax(Q_hybsrc) =", full_greedy(Q_hybsrc), " (picks action 2 at s=0)")
V_g = evaluate_scalar(full_greedy(Qhat), nxt, P, phi_w, gamma, n)
V_h = evaluate_scalar(full_greedy(Q_hybsrc), nxt, P, phi_w, gamma, n)
print(f"realized value gap from this single flipped tie-break: {np.max(np.abs(V_g - V_h)):.4f}")
print()
print("This is why the stratified sweep below jitters phi by 1e-3: it turns exact ties")
print("into a generic (measure-zero-excluded) strict order, so genuine k=1 violation")
print("counts drop to exactly 0/150 in every configuration.")


Demonstrating the floating-point tie artifact at k=1 (a fixed, clean, non-jittered example)

row 0 of Qhat,     full precision: [0.6258726962083455 0.563285426587511  0.6258726962083455
 0.6258726962083455]
row 0 of Q_hybsrc, full precision: [0.6258726962083456 0.5632854265875111 0.6258726962083456
 0.6258726962083456]
Same real-valued function (Lemma 1) -- a genuine 3-way tie among actions {0,2,3}
at value 0.62587... -- but rounded via two different linear solves, so argmax flips:
argmax(Qhat)     = [0 3 0 1 3 2]  (picks action 0 at s=0)
argmax(Q_hybsrc) = [0 3 0 1 3 2]  (picks action 2 at s=0)
realized value gap from this single flipped tie-break: 0.0000

This is why the stratified sweep below jitters phi by 1e-3: it turns exact ties
into a generic (measure-zero-excluded) strict order, so genuine k=1 violation
counts drop to exactly 0/150 in every configuration.


## 4. Stratified random-MDP sweep, $k=1,\dots,4$

150 trials per cell (deterministic/stochastic $\times$ dense/sparse reward $\times$ $k$),
8 random query weights per trial, sizes $n\in[3,8]$, $m\in[3,5]$, $d\in\{2,3\}$.

In [4]:
import time
import zlib


def seed_of(tag):
    return zlib.crc32(tag.encode()) % (2**32)


def run_trial(rng, n, m, d, k, gamma, deterministic, sparse_reward, n_query_weights=8):
    nxt, P, phi = make_mdp(n, m, d, rng, deterministic, sparse_reward)
    base_pis = []
    for _ in range(k):
        w_train = rng.dirichlet(np.ones(d))
        pi_j, _ = policy_iteration(phi @ w_train, nxt, P, gamma, n, m)
        base_pis.append(pi_j)
    worst = 0.0
    for _ in range(n_query_weights):
        w = rng.dirichlet(np.ones(d))
        out = solve_basis_and_compare(nxt, P, phi, base_pis, w, gamma, n, m, d)
        worst = max(worst, np.max(np.abs(out["V_hyb"] - out["V_gpi"])))
    return worst


gamma = GAMMA
n_trials_per_cell = 150
k_values = [1, 2, 3, 4]
TOL = 1e-6
results = {}

t0 = time.perf_counter()
for deterministic in [True, False]:
    for sparse_reward in [True, False]:
        for k in k_values:
            tag = f"det={deterministic} sparse={sparse_reward} k={k}"
            max_diff, violations = 0.0, 0
            rng_master = np.random.default_rng(seed_of(tag))
            for _ in range(n_trials_per_cell):
                n = int(rng_master.integers(max(3, k + 2), 9))
                m = int(rng_master.integers(max(3, k + 1), 6))
                d = int(rng_master.integers(2, 4))
                rng = np.random.default_rng(rng_master.integers(0, 2**31))
                worst = run_trial(rng, n, m, d, k, gamma, deterministic, sparse_reward)
                max_diff = max(max_diff, worst)
                violations += worst > TOL
            results[tag] = (max_diff, violations, n_trials_per_cell)
            print(f"{tag:35s} max|V_hyb-V_gpi|={max_diff:8.4f}  violations={violations:3d}/{n_trials_per_cell}")
print(f"\n({time.perf_counter()-t0:.0f}s)")


det=True sparse=True k=1            max|V_hyb-V_gpi|=  0.0000  violations=  0/150


det=True sparse=True k=2            max|V_hyb-V_gpi|=  1.5811  violations= 13/150


det=True sparse=True k=3            max|V_hyb-V_gpi|=  2.9902  violations= 20/150


det=True sparse=True k=4            max|V_hyb-V_gpi|=  4.4728  violations= 37/150


det=True sparse=False k=1           max|V_hyb-V_gpi|=  0.0000  violations=  0/150


...

det=False sparse=True k=4           max|V_hyb-V_gpi|=  0.7471  violations= 50/150


det=False sparse=False k=1          max|V_hyb-V_gpi|=  0.0000  violations=  0/150


det=False sparse=False k=2          max|V_hyb-V_gpi|=  0.4309  violations= 26/150


det=False sparse=False k=3          max|V_hyb-V_gpi|=  0.4284  violations= 48/150


det=False sparse=False k=4          max|V_hyb-V_gpi|=  0.8548  violations= 52/150

(77s)


## 5. Minimal counterexample #1: $\pi^{\mathrm{hyb}}$ strictly beats $\pi^{\mathrm{GPI}}_{\mathcal B}$

Mined down to the smallest case found (3 states, 3 actions, 2 objectives, $k=2$) and
verified independently by hand (see `notes/adaptive_basis.tex`). The mechanism: a
self-loop action, unreachable to the metapolicy because it belongs to neither base
policy's own action at that state, turns out to be a better *permanent* choice than
either base policy's trajectory -- and $\pi^{\mathrm{hyb}}$'s bootstrap off the already-composed
$V_{\mathrm{meta}}$ (rather than off any single base policy's own, weaker, continuation)
is exactly what lets it see that.

In [5]:
print("Minimal counterexample #1: pi_hyb STRICTLY BEATS pi_gpi_B (n=3, m=3, d=2, k=2)")
print()

nxt = np.array([[0, 0, 1], [2, 0, 0], [1, 1, 1]])
phi = np.array([
    [[0.19788131, 0.76069892], [0.72670145, 0.50731725], [0.27641115, 0.21776944]],
    [[0.70319209, 0.56015840], [0.53220768, 0.13021757], [0.26588580, 0.69896058]],
    [[0.98280998, 0.06645318], [0.72847186, 0.63121623], [0.75073821, 0.08131848]],
])
w = np.array([0.15245458, 0.84754542])
base_pis = [np.array([1, 0, 1]), np.array([2, 0, 0])]  # pi_1, pi_2
n, m, d, gamma = 3, 3, 2, GAMMA

out = solve_basis_and_compare(nxt, None, phi, base_pis, w, gamma, n, m, d)
print("B(0)={1,2}, B(1)={0}, B(2)={0,1}  (the two base policies' own actions per state)")
print("pi_meta =", out["pi_meta"], "  pi_gpi =", out["pi_gpi"], "  pi_hyb =", out["pi_hyb"])
print("V_meta  =", out["V_meta"])
print("V_gpi   =", out["V_gpi"], " (= V_meta here: GPI happens to reproduce the metapolicy)")
print("V_hyb   =", out["V_hyb"])
print()
print("pi_hyb(0)=0 is a self-loop (0 -> 0) NOT in B(0)={1,2}: the metapolicy could never")
print("choose it. Its one-step lookahead value, bootstrapped off V_meta(0), is")
print(f"  phi(0,0).w + gamma*V_meta(0) = {phi[0,0]@w:.4f} + {gamma}*{out['V_meta'][0]:.4f}"
      f" = {phi[0,0]@w + gamma*out['V_meta'][0]:.4f}")
print(f"which exceeds V_meta(0)={out['V_meta'][0]:.4f} itself, so pi_hyb takes it -- and since")
print("it is a genuine self-loop, the REALIZED value is the resulting geometric series:")
print(f"  phi(0,0).w / (1-gamma) = {phi[0,0]@w:.4f} / {1-gamma} = {phi[0,0]@w/(1-gamma):.4f}")
print(f"matching V_hyb(0)={out['V_hyb'][0]:.4f}, strictly above V_gpi(0)={out['V_gpi'][0]:.4f}.")
print()
Psi1, _ = full_sf_table(base_pis[0], nxt, None, phi, gamma, n, m, d)
Psi2, _ = full_sf_table(base_pis[1], nxt, None, phi, gamma, n, m, d)
print("Why GPI misses this action: its pool estimate for action 0 at state 0 bootstraps off")
print("EACH base policy's OWN (mediocre) continuation, not off the composed metapolicy's:")
print(f"  Psi_pi1(0,0).w = {Psi1[0,0]@w:.4f}   Psi_pi2(0,0).w = {Psi2[0,0]@w:.4f}"
      f"  ->  Qhat(0,0) = {max(Psi1[0,0]@w, Psi2[0,0]@w):.4f}")
print(f"  Qhat(0, pi_gpi(0)={out['pi_gpi'][0]}) = {out['Qhat'][0, out['pi_gpi'][0]]:.4f}"
      f"  (this wins the pool's own argmax)")
print(f"  Q_hybsrc(0,0) = {out['Q_hybsrc'][0,0]:.4f}  (bootstraps off V_meta instead -- higher)")


Minimal counterexample #1: pi_hyb STRICTLY BEATS pi_gpi_B (n=3, m=3, d=2, k=2)

B(0)={1,2}, B(1)={0}, B(2)={0,1}  (the two base policies' own actions per state)
pi_meta = [2 0 1]   pi_gpi = [2 0 1]   pi_hyb = [0 0 1]
V_meta  = [5.7376 6.1232 6.1569]
V_gpi   = [5.7376 6.1232 6.1569]  (= V_meta here: GPI happens to reproduce the metapolicy)
V_hyb   = [6.7489 6.1232 6.1569]

pi_hyb(0)=0 is a self-loop (0 -> 0) NOT in B(0)={1,2}: the metapolicy could never
choose it. Its one-step lookahead value, bootstrapped off V_meta(0), is
  phi(0,0).w + gamma*V_meta(0) = 0.6749 + 0.9*5.7376 = 5.8387
which exceeds V_meta(0)=5.7376 itself, so pi_hyb takes it -- and since
it is a genuine self-loop, the REALIZED value is the resulting geometric series:
  phi(0,0).w / (1-gamma) = 0.6749 / 0.09999999999999998 = 6.7489
matching V_hyb(0)=6.7489, strictly above V_gpi(0)=5.7376.

Why GPI misses this action: its pool estimate for action 0 at state 0 bootstraps off
EACH base policy's OWN (mediocre) continuation, 

## 6. Minimal counterexample #2: the reverse direction

Neither policy dominates the other: this $3\times4\times2$, $k=2$ example has
$\pi^{\mathrm{GPI}}_{\mathcal B}$ strictly ahead of $\pi^{\mathrm{hyb}}$ instead.

In [6]:
print("Minimal counterexample #2: pi_gpi_B STRICTLY BEATS pi_hyb (n=3, m=4, d=2, k=2)")
print()

nxt = np.array([[2, 2, 2, 1], [1, 1, 0, 1], [1, 2, 0, 2]])
phi = np.array([
    [[0.31898189, 0.12647578], [0.49033860, 0.74536444], [0.01150748, 0.13755603], [0.63719606, 0.58878224]],
    [[0.02694831, 0.74838586], [0.40844423, 0.88605940], [0.37537800, 0.25121873], [0.82470631, 0.03710746]],
    [[0.74559804, 0.17799423], [0.61045478, 0.06783933], [0.85616052, 0.82155972], [0.74909825, 0.50135312]],
])
w = np.array([0.68023705, 0.31976295])
base_pis = [np.array([3, 3, 0]), np.array([3, 1, 2])]  # pi_1, pi_2
n, m, d, gamma = 3, 4, 2, GAMMA

out = solve_basis_and_compare(nxt, None, phi, base_pis, w, gamma, n, m, d)
print("B(0)={3}, B(1)={1,3}, B(2)={0,2}")
print("pi_meta =", out["pi_meta"], "  pi_gpi =", out["pi_gpi"], "  pi_hyb =", out["pi_hyb"])
print("V_meta  =", out["V_meta"])
print("V_gpi   =", out["V_gpi"])
print("V_hyb   =", out["V_hyb"])
print()
print(f"At s=0: V_gpi(0)={out['V_gpi'][0]:.4f} > V_hyb(0)={out['V_hyb'][0]:.4f}, gap ="
      f" {out['V_gpi'][0]-out['V_hyb'][0]:.4f}.")
print()
print("pi_gpi and pi_hyb agree at s=0 and s=1 (both pick action 1, then action 3); they")
print(f"diverge only at s=2: pi_gpi(2)={out['pi_gpi'][2]} (= pi_meta(2), inside B(2)={{0,2}})")
print(f"vs. pi_hyb(2)={out['pi_hyb'][2]} (outside B(2)). Since s=0's chosen action already")
print("transitions straight to s=2 (nxt(0,1)=2), this single downstream mistake at s=2 is")
print("enough to flip the total realized value all the way back at the start state -- the")
print("hybrid bootstrap's one-step lookahead is not always the right one to trust at every")
print("state along the resulting trajectory. Net effect: neither pi_hyb nor pi_gpi_B")
print("dominates the other in general -- Theorem (tie fails at k>=2) holds in both directions.")


Minimal counterexample #2: pi_gpi_B STRICTLY BEATS pi_hyb (n=3, m=4, d=2, k=2)

B(0)={3}, B(1)={1,3}, B(2)={0,2}
pi_meta = [3 3 2]   pi_gpi = [1 3 2]   pi_hyb = [1 3 3]
V_meta  = [5.7775 5.7286 6.0448]
V_gpi   = [7.013  5.7286 7.1568]
V_hyb   = [6.6008 5.7286 6.6988]

At s=0: V_gpi(0)=7.0130 > V_hyb(0)=6.6008, gap = 0.4122.

pi_gpi and pi_hyb agree at s=0 and s=1 (both pick action 1, then action 3); they
diverge only at s=2: pi_gpi(2)=2 (= pi_meta(2), inside B(2)={0,2})
vs. pi_hyb(2)=3 (outside B(2)). Since s=0's chosen action already
transitions straight to s=2 (nxt(0,1)=2), this single downstream mistake at s=2 is
enough to flip the total realized value all the way back at the start state -- the
hybrid bootstrap's one-step lookahead is not always the right one to trust at every
state along the resulting trajectory. Net effect: neither pi_hyb nor pi_gpi_B
dominates the other in general -- Theorem (tie fails at k>=2) holds in both directions.


## 7. Does "terminate on pickup" (Office/DST/the maze's structure) restore the tie?

Same generator, but every sparse-reward event now transitions to an explicit absorbing
zero-reward sink instead of continuing in the random walk -- mirroring every navigation
domain tested so far.

In [7]:
import zlib


def seed_of(tag):
    return zlib.crc32(tag.encode()) % (2**32)


def make_terminating_mdp(n_live, m, d, rng, deterministic, jitter=1e-3):
    """n_live live states plus one absorbing zero-reward sink, mirroring the
    "terminate on reward pickup" structure of DST/Office/the maze -- unlike
    make_mdp's infinite-horizon random walk, a reward event here ends the episode."""
    n = n_live + 1
    sink = n_live
    if deterministic:
        nxt = rng.integers(0, n_live, size=(n, m))
        nxt[sink, :] = sink
        P = None
    else:
        alpha = 0.3
        P = np.zeros((n, m, n))
        for s in range(n_live):
            for a in range(m):
                P[s, a, :n_live] = rng.dirichlet(alpha * np.ones(n_live))
        P[sink, :, sink] = 1.0
        nxt = None

    phi = np.zeros((n, m, d))
    mask = rng.random((n_live, m)) < 0.25
    idx = rng.integers(0, d, size=(n_live, m))
    for s in range(n_live):
        for a in range(m):
            if mask[s, a]:
                phi[s, a, idx[s, a]] = 1.0
                if deterministic:
                    nxt[s, a] = sink
                else:
                    P[s, a, :] = 0.0
                    P[s, a, sink] = 1.0
    phi[:n_live] += jitter * rng.random((n_live, m, d))
    return nxt, P, phi


def run_trial_terminating(rng, n_live, m, d, k, gamma, deterministic, n_query_weights=8):
    nxt, P, phi = make_terminating_mdp(n_live, m, d, rng, deterministic)
    n = n_live + 1
    base_pis = []
    for _ in range(k):
        w_train = rng.dirichlet(np.ones(d))
        pi_j, _ = policy_iteration(phi @ w_train, nxt, P, gamma, n, m)
        base_pis.append(pi_j)
    worst = 0.0
    for _ in range(n_query_weights):
        w = rng.dirichlet(np.ones(d))
        out = solve_basis_and_compare(nxt, P, phi, base_pis, w, gamma, n, m, d)
        worst = max(worst, np.max(np.abs(out["V_hyb"] - out["V_gpi"])))
    return worst


print("Hypothesis: DST/Office/the maze's zero counterexamples come from their absorbing,")
print("terminate-on-pickup structure, not a general k>=2 guarantee. Test: add an explicit")
print("absorbing sink on every reward event and re-run the sweep.")
print()
gamma = GAMMA
n_trials = 150
for deterministic in [True, False]:
    for k in [1, 2, 3, 4]:
        tag = f"TERMINATING det={deterministic} k={k}"
        max_diff, violations = 0.0, 0
        rng_master = np.random.default_rng(seed_of(tag))
        for _ in range(n_trials):
            n_live = int(rng_master.integers(max(3, k + 2), 8))
            m = int(rng_master.integers(max(3, k + 1), 6))
            d = int(rng_master.integers(2, 4))
            rng = np.random.default_rng(rng_master.integers(0, 2**31))
            worst = run_trial_terminating(rng, n_live, m, d, k, gamma, deterministic)
            max_diff = max(max_diff, worst)
            violations += worst > 1e-6
        print(f"{tag:25s} max|V_hyb-V_gpi|={max_diff:8.4f}  violations={violations:3d}/{n_trials}")


Hypothesis: DST/Office/the maze's zero counterexamples come from their absorbing,
terminate-on-pickup structure, not a general k>=2 guarantee. Test: add an explicit
absorbing sink on every reward event and re-run the sweep.



TERMINATING det=True k=1  max|V_hyb-V_gpi|=  0.0000  violations=  0/150


TERMINATING det=True k=2  max|V_hyb-V_gpi|=  0.0412  violations=  2/150


TERMINATING det=True k=3  max|V_hyb-V_gpi|=  0.6999  violations=  1/150


TERMINATING det=True k=4  max|V_hyb-V_gpi|=  0.5077  violations=  7/150


TERMINATING det=False k=1 max|V_hyb-V_gpi|=  0.0000  violations=  0/150


TERMINATING det=False k=2 max|V_hyb-V_gpi|=  0.3449  violations= 13/150


TERMINATING det=False k=3 max|V_hyb-V_gpi|=  0.1390  violations= 29/150


TERMINATING det=False k=4 max|V_hyb-V_gpi|=  0.4325  violations= 34/150


## Conclusion

The tie is an exact theorem at $k=1$ and false in general at $k\ge2$, in both directions.
Absorbing/terminating structure suppresses the failure rate by roughly an order of
magnitude but does not eliminate it, which is consistent with Office ($k\le6$) and the
maze ($k\le2$) simply not having sampled the (rare, $\sim1$-$2\%$) violating region in
their $150$-weight tests, rather than any structural guarantee. See
`notes/adaptive_basis.tex` for the formal Lemma/Theorem statements and proofs built on
these findings.